In [1]:
from langchain_groq import ChatGroq
from langgraph.graph import StateGraph, START, END
from typing import TypedDict
from dotenv import load_dotenv

In [2]:
load_dotenv()
model = ChatGroq(model="llama-3.1-8b-instant",temperature=1.5)

In [3]:
class BatsmanState(TypedDict):
    # inputs
    runs: int
    balls: int
    fours: int
    sixes: int
    # outputs
    sr: float
    bpb:float
    boundary_percent: float 
    summary:str

In [4]:
def calculate_sr(state: BatsmanState) -> BatsmanState:
    sr = (state['runs'] / state['balls']) * 100
    return {'sr': sr}

In [5]:
def calculate_bpb(state: BatsmanState) -> BatsmanState:
    bpb = (state['balls'] / state['fours']+state['sixes']) 
    return {'bpb':bpb}

In [6]:
def calculate_boundary_percent(state: BatsmanState) -> BatsmanState:
    boundary_percent = ((state['fours']*4+state['sixes']*6) / state['runs'])*100
    return {'boundary_percent':boundary_percent}

In [7]:
def summary(state: BatsmanState) -> BatsmanState:
    summary = f"""
    Strike Rate - {state['sr']}\n 
    Balls per boundary - {state['bpb']}\n 
    Boundary percent - {state['boundary_percent']}
    """

    return {'summary': summary}

In [8]:
graph = StateGraph(BatsmanState)

graph.add_node('calculate_sr',calculate_sr)
graph.add_node('calculate_bpb',calculate_bpb)
graph.add_node('calculate_boundary_percent',calculate_boundary_percent)
graph.add_node('summary',summary)

graph.add_edge(START,'calculate_sr')
graph.add_edge(START,'calculate_bpb')
graph.add_edge(START,'calculate_boundary_percent')

graph.add_edge('calculate_sr','summary')
graph.add_edge('calculate_bpb','summary')
graph.add_edge('calculate_boundary_percent','summary')

graph.add_edge('summary',END)

workflow = graph.compile()
# we get error when run parallel workflow bcoz they think that there is parallel changes  which is not allowed.
# InvalidUpdateError: At key 'runs': Can receive only one value per step. Use an Annotated key to handle multiple values.

# to solve this error return a single state in methods or functions


In [9]:
inital_state = {
    'runs':100,
    'balls': 50,
    'fours': 5,
    'sixes': 2,
}

workflow.invoke(inital_state)

{'runs': 100,
 'balls': 50,
 'fours': 5,
 'sixes': 2,
 'sr': 200.0,
 'bpb': 12.0,
 'boundary_percent': 32.0,
 'summary': '\n    Strike Rate - 200.0\n \n    Balls per boundary - 12.0\n \n    Boundary percent - 32.0\n    '}